# Name:Jakiel
# Date: 11-23-25
# Objective:Our goal of the project is to build a model that predicts the quality of wine by using complex ensemble modeling.

In [107]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    BaggingClassifier,
    VotingClassifier,)
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,)

# Section 1. Load and Inspect the Data


In [108]:
#Load wine dataset and display first
wine = pd.read_csv("winequality-red.csv", sep=";")

# Display and first 10 rows
wine.head(10)

# The dataset includes 11 physicochemical input variables (features):
# ---------------------------------------------------------------
# - fixed acidity          mostly tartaric acid
# - volatile acidity       mostly acetic acid (vinegar)
# - citric acid            can add freshness and flavor
# - residual sugar         remaining sugar after fermentation
# - chlorides              salt content
# - free sulfur dioxide    protects wine from microbes
# - total sulfur dioxide   sum of free and bound forms
# - density                related to sugar content
# - pH                     acidity level (lower = more acidic)
# - sulphates              antioxidant and microbial stabilizer
# - alcohol                % alcohol by volume


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
5,7.4,0.66,0.00,1.8,0.075,13.0,40.0,0.9978,3.51,0.56,9.4,5
6,7.9,0.60,0.06,1.6,0.069,15.0,59.0,0.9964,3.30,0.46,9.4,5
7,7.3,0.65,0.00,1.2,0.065,15.0,21.0,0.9946,3.39,0.47,10.0,7
8,7.8,0.58,0.02,2.0,0.073,9.0,18.0,0.9968,3.36,0.57,9.5,7
9,7.5,0.50,0.36,6.1,0.071,17.0,102.0,0.9978,3.35,0.80,10.5,5


In [109]:
# Display structure of dataset
wine.info

<bound method DataFrame.info of       fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0               7.4             0.700         0.00             1.9      0.076   
1               7.8             0.880         0.00             2.6      0.098   
2               7.8             0.760         0.04             2.3      0.092   
3              11.2             0.280         0.56             1.9      0.075   
4               7.4             0.700         0.00             1.9      0.076   
...             ...               ...          ...             ...        ...   
1594            6.2             0.600         0.08             2.0      0.090   
1595            5.9             0.550         0.10             2.2      0.062   
1596            6.3             0.510         0.13             2.3      0.076   
1597            5.9             0.645         0.12             2.0      0.075   
1598            6.0             0.310         0.47             3.6      0.067

In [110]:
print('The target variable is quality.Score ranges from 0 to 10, rated by wine tasters.')
print('We will simplify this target into three categories:low (3–4), medium (5–6), high (7–8) to make classification feasible.')

The target variable is quality.Score ranges from 0 to 10, rated by wine tasters.
We will simplify this target into three categories:low (3–4), medium (5–6), high (7–8) to make classification feasible.


# Section 2. Prepare the Data


In [111]:
# Create quality_label feature that returns low medium high when scores fall within a certain range.
def quality_to_label(q):
    if q <= 4:
        return "low"
    elif q <= 6:
        return "medium"
    else:
        return "high"

#Call the apply() method on the quality column to create the new quality_label column
wine["quality_label"] = wine["quality"].apply(quality_to_label)

#Show newly created column
wine.quality_label


0       medium
1       medium
2       medium
3       medium
4       medium
         ...  
1594    medium
1595    medium
1596    medium
1597    medium
1598    medium
Name: quality_label, Length: 1599, dtype: object

In [112]:
#Create a numeric column for modeling: 0 = low, 1 = medium, 2 = high
def quality_to_number(q):
    if q <= 4:
        return 0
    elif q <= 6:
        return 1
    else:
        return 2

#Call the apply() method on the quality column to create the new quality_number column
wine["quality_number"] = wine["quality"].apply(quality_to_number)

#Show newly created column
wine.quality_number


0       1
1       1
2       1
3       1
4       1
       ..
1594    1
1595    1
1596    1
1597    1
1598    1
Name: quality_number, Length: 1599, dtype: int64

# Section 3. Feature Selection and Justification


In [113]:
# I will use all columns execpt target (qaulity) and ones I created ('quality' and 'quality label').

# Drop quality, quality label and quality number columns

#features
X = wine.drop(columns=["quality","quality_label","quality_number"])

#target
y = wine["quality_number"]

# Section 4. Split the Data into Train and Test


In [114]:
# Train/test split(stratify to keep class balance)
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=.2,random_state=42,stratify=y)

# Section 5.  Evaluate Model Performance


In [115]:
def evaluate_model(name, model, X_train, y_train, X_test, y_test, results):
    
    # Train the model
    model.fit(X_train, y_train)

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Calculate metrics
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    train_f1 = f1_score(y_train, y_train_pred, average="weighted")
    test_f1 = f1_score(y_test, y_test_pred, average="weighted")

    # Print results
    print(f"\n{name} Results")
    print("Confusion Matrix (Test):")
    print(confusion_matrix(y_test, y_test_pred))
    print(f"Train Accuracy: {train_acc:.4f}, Test Accuracy: {test_acc:.4f}")
    print(f"Train F1 Score: {train_f1:.4f}, Test F1 Score: {test_f1:.4f}")

    # Store results
    results.append({
            "Model": name,
            "Train Accuracy": train_acc,
            "Test Accuracy": test_acc,
            "Train F1": train_f1,
            "Test F1": test_f1,})

    # Initialize the results container
results = [] 

In [121]:
#Voting Classifier
voting1 = VotingClassifier(
    estimators=[
        ("DT", DecisionTreeClassifier()),
        ("SVM", SVC(probability=True)),
        ("NN", MLPClassifier(hidden_layer_sizes=(50,), max_iter=1000)),
    ],
    voting="soft",
)
evaluate_model(
    "Voting (DT + SVM + NN)", voting1, X_train, y_train, X_test, y_test, results
)


Voting (DT + SVM + NN) Results
Confusion Matrix (Test):
[[  0  12   1]
 [  0 247  17]
 [  0  19  24]]
Train Accuracy: 0.9195, Test Accuracy: 0.8469
Train F1 Score: 0.9000, Test F1 Score: 0.8278


In [117]:
# MLP Classifier 
evaluate_model(
    "MLP Classifier",
    MLPClassifier(hidden_layer_sizes=(100,), max_iter=1000, random_state=42),
    X_train,
    y_train,
    X_test,
    y_test,
    results,
)


MLP Classifier Results
Confusion Matrix (Test):
[[  0  13   0]
 [  0 257   7]
 [  0  30  13]]
Train Accuracy: 0.8514, Test Accuracy: 0.8438
Train F1 Score: 0.8141, Test F1 Score: 0.8073


# Section 6. Comparing Results

In [119]:
#Create a table of results
results_wine = pd.DataFrame(results)

print("\nSummary of Models:")
display(results_wine)


Summary of Models:


,Model,Train Accuracy,Test Accuracy,Train F1,Test F1
0,Voting (DT + SVM + NN),0.923378,0.859375,0.906188,0.839221
1,MLP Classifier,0.851446,0.843750,0.814145,0.807318
2,"Random Forest (200, max_depth=10)",0.975762,0.881250,0.974482,0.859643


# Section 7. Conclusions and Insights


Since our objective was to predict the quality of wine. I converted the quality column which hosts a score of 0 to 10 rated by consumers into 2 more columns, quality label which hosts low, medium or high and quality number that hosts 0,1 or 2. I decided to compare the Voting classifier with the MLP classifier. The Voting model combines predictions from a decision tree, SVM, and neural network, so I wanted to see whether this ensemble approach would give more stable predictions for wine quality.
The MLP reached a test accuracy of 0.8438 with an F1 score of 0.8073. The accuracy gap between training and testing was only 0.0076, which shows the model was very consistent and not overfitting. However, the confusion matrix revealed a major limitation it completely missed the minority class and didn’t correctly predict any samples from the lowest quality level. So even though the model was stable, there’s still room to improve its predictive.
When I tried the Voting ensemble, the results were slightly better. Its test accuracy was 0.8469 and its F1 score was 0.8278. It has a larger accuracy gap of about 0.0726, which suggests a bit more overfitting. Still, it performed better overall than the MLP. Both models struggled with the minority class, but I would choose the Voting classifier because it delivered higher accuracy and F1 scores for the wine quality levels it did identify correctly.